In [1]:
import h2o
print(h2o.__version__)

3.46.0.9


In [24]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
import re
import warnings
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ML
import h2o
from h2o.automl import H2OAutoML

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    root_mean_squared_error = None

import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.29.7-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]



In [27]:
class MercariFullPipeline:

    def __init__(self, data_dir="../data", images_dir="../images", results_dir="../results"):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        self.train = None
        self.test = None
        self.train_vectorized = None
        self.test_vectorized = None

        self.train_hf = None
        self.test_hf = None
        self.automl = None
        self.best_model = None
        self.leaderboard_df = None
        self.feature_cols = None
        self.metrics = {}

        # 추가: LGBM + Optuna + SHAP 모델 저장용
        self.lgbm_model = None
        self.lgbm_best_params = None

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)



    # ----------------- 유틸 함수 -----------------

    def _simple_normalize(self, text: str) -> str:
        text = str(text).lower()
        text = re.sub(r"[_\-\./]", " ", text)
        text = re.sub(r"\d+", " num ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        combined = pd.concat([self.train[col], self.test[col]], axis=0)
        value_counts = combined.value_counts()
        top_values = value_counts.index[:top_k]

        self.train[col] = self.train[col].where(self.train[col].isin(top_values), rare_label)
        self.test[col] = self.test[col].where(self.test[col].isin(top_values), rare_label)



    # ----------------- 1. 데이터 로딩 -----------------

    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t"):
        print("📂 Loading...")
        train_path = os.path.join(self.data_dir, train_file)
        test_path = os.path.join(self.data_dir, test_file)

        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        # Remove invalid prices
        self.train = self.train[self.train["price"] > 0]

        # Split category
        for df_name, df in [("train", self.train), ("test", self.test)]:
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (
                        x.split("/") if isinstance(x, str) and "/" in x else ["missing", "missing", "missing"]
                    )
                )
            )
            df["name"] = df["name"].fillna("unknown")
            df["brand_name"] = df["brand_name"].fillna("Unknown")
            df["item_description"] = df["item_description"].fillna("No desc")

        # Log transform price
        self.train["price"] = np.log1p(self.train["price"].clip(1, 2500))

        # Collapse rare
        self._collapse_rare_values("brand_name", 2000, "Other_brand")
        self._collapse_rare_values("sub_cat", 300)
        self._collapse_rare_values("sub_sub_cat", 300)

        print("📌 Load COMPLETE!")



    # ----------------- 2. 벡터화 -----------------

    def vectorize_text(self, text_columns=["name", "item_description"], method="tfidf", max_features=40000, n_components=80):
        print("🔤 Vectorizing...")
        vectors = []
        names = []

        for col in tqdm(text_columns):
            self.train[f"{col}_clean"] = self.train[col].astype(str).apply(self._simple_normalize)
            self.test[f"{col}_clean"] = self.test[col].astype(str).apply(self._simple_normalize)

            vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2)) if method=="tfidf" else CountVectorizer(max_features=max_features)

            combined = pd.concat([self.train[f"{col}_clean"], self.test[f"{col}_clean"]])
            vec.fit(combined)

            train_vec = vec.transform(self.train[f"{col}_clean"])
            test_vec = vec.transform(self.test[f"{col}_clean"])

            svd = TruncatedSVD(n_components=n_components, random_state=23)
            vectors.append((svd.fit_transform(train_vec), svd.transform(test_vec)))
            names.append([f"{col}_{i}" for i in range(n_components)])

        X_train = np.hstack([v[0] for v in vectors])
        X_test = np.hstack([v[1] for v in vectors])

        self.train_vectorized = pd.DataFrame(X_train, columns=[x for sub in names for x in sub])
        self.test_vectorized = pd.DataFrame(X_test, columns=[x for sub in names for x in sub])

        # Add categorical & numeric features
        extra_features = ["main_cat", "sub_cat", "sub_sub_cat", "brand_name", "shipping", "item_condition_id", "name_len_word", "desc_len_word"]
        self.train["name_len_word"] = self.train["name"].str.split().str.len()
        self.train["desc_len_word"] = self.train["item_description"].str.split().str.len()

        self.test["name_len_word"] = self.test["name"].str.split().str.len()
        self.test["desc_len_word"] = self.test["item_description"].str.split().str.len()

        for col in extra_features:
            self.train_vectorized[col] = self.train[col]
            self.test_vectorized[col] = self.test[col]

        print("📌 Vectorization COMPLETE:", self.train_vectorized.shape)



    # ----------------- 3. H2O AutoML Training -----------------

    def init_h2o(self, max_mem_size="8G", nthreads=-1):
        print(f"🚀 Initializing H2O (Memory={max_mem_size}, Threads={nthreads})")
        h2o.init(max_mem_size=max_mem_size, nthreads=nthreads)


    def train_automl(self, max_models=15, metric="RMSE", seed=23, nfolds=5):
        df = self.train_vectorized.copy()
        df["price"] = self.train["price"].values

        self.train_hf = h2o.H2OFrame(df)
        self.test_hf = h2o.H2OFrame(self.test_vectorized)

        # Factor 처리
        for c in ["main_cat","sub_cat","sub_sub_cat","brand_name","shipping","item_condition_id"]:
            self.train_hf[c] = self.train_hf[c].asfactor()
            self.test_hf[c] = self.test_hf[c].asfactor()

        self.feature_cols = [c for c in self.train_hf.columns if c!="price"]

        self.automl = H2OAutoML(
            max_models=max_models,
            seed=seed,
            sort_metric=metric,
            nfolds=nfolds
        )

        self.automl.train(
            x=self.feature_cols,
            y="price",
            training_frame=self.train_hf
        )

        self.best_model = self.automl.leader
        self.leaderboard_df = self.automl.leaderboard.as_data_frame()
        print(self.leaderboard_df.head())




    # ----------------- 4. Force Stacked Ensemble -----------------

    def use_stacked_ensemble_as_leader(self):
        lb = self.leaderboard_df

        se = lb[lb["model_id"].str.contains("StackedEnsemble")]
        if len(se) > 0:
            best = se.iloc[0]["model_id"]
            self.best_model = h2o.get_model(best)
            print(f"🔥 Leader replaced with Stacked Ensemble: {best}")
        else:
            print("⚠ No Stacked Ensemble found.")



    # ----------------- 5. LightGBM + Optuna -----------------

    def train_lgbm_optuna(self, n_trials=20):
        import optuna, lightgbm as lgb
        from sklearn.model_selection import train_test_split

        X = self.train_vectorized.copy()
        y_log = self.train["price"].values
        y_true = np.expm1(y_log)

        X_train, X_valid, y_train_log, y_valid_log, y_train_true, y_valid_true = train_test_split(
            X, y_log, y_true, test_size=0.2, random_state=23
        )

        train_set = lgb.Dataset(X_train, label=y_train_log)
        valid_set = lgb.Dataset(X_valid, label=y_valid_log)

        def objective(trial):
            params = {
                "objective": "regression",
                "metric": "rmse",
                "learning_rate": trial.suggest_float("lr", 0.01, 0.2),
                "num_leaves": trial.suggest_int("num_leaves", 31, 255),
            }
            model = lgb.train(params, train_set, valid_sets=[valid_set], callbacks=[lgb.early_stopping(30)])
            preds = np.expm1(model.predict(X_valid))
            return mean_squared_error(y_valid_true, preds, squared=False)

        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=n_trials)
        print("Optuna Best:", study.best_value, study.best_params)

        params = {"objective": "regression", "metric": "rmse"}
        params.update(study.best_params)
        self.lgbm_model = lgb.train(params, lgb.Dataset(X, label=y_log))
        self.lgbm_best_params = params



    # ----------------- 6. SHAP -----------------

    def analyze_shap(self, sample=3000):
        import shap

        X = self.train_vectorized.head(sample)
        explainer = shap.TreeExplainer(self.lgbm_model)
        shap_values = explainer.shap_values(X)

        print("🔥 SHAP Top Features:")
        importance = np.abs(shap_values).mean(0)
        idx = np.argsort(importance)[::-1][:20]

        for i in idx:
            print(f"{X.columns[i]}   → {importance[i]:.4f}")



    # ----------------- 7. Submission -----------------

    def predict_test(self, file="submission.csv"):
        preds = self.best_model.predict(self.test_hf).as_data_frame()['predict'].values
        preds = np.expm1(preds)

        sub = pd.DataFrame({"test_id": self.test["test_id"], "price": preds})
        path = os.path.join(self.results_dir, file)
        sub.to_csv(path, index=False)
        print(f"📁 Saved: {path}")
        return sub

In [4]:
# ------------------ 실행 예시 ------------------


    
# MercariFullPipeline()

# load_data()
# vectorize_text()

# init_h2o()
# train_automl()
# use_stacked_ensemble_as_leader()

# train_lgbm_optuna(n_trials=20)
# analyze_shap()

# predict_test()


In [28]:
analyzer = MercariFullPipeline(
    data_dir="../data",
    images_dir="../images",
    results_dir="../results"
)


In [29]:
analyzer.load_data()

📂 Loading...
📌 Load COMPLETE!


In [30]:
analyzer.vectorize_text(
    method="tfidf",
    max_features=35000,
    n_components=80
)

🔤 Vectorizing...


100%|██████████| 2/2 [07:27<00:00, 223.62s/it]


📌 Vectorization COMPLETE: (1481661, 168)


In [31]:
analyzer.init_h2o(max_mem_size="12G")
# analyzer.init_h2o(max_mem_size="16G") -> 에러났을 때 이걸로 변경

🚀 Initializing H2O (Memory=12G, Threads=-1)
Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,10 mins 52 secs
H2O_cluster_timezone:,Asia/Seoul
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,10 days
H2O_cluster_name:,H2O_from_python_tj_pdnzxs
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,12 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"


In [36]:
import h2o
print(h2o.cluster_info())

H2O_cluster_uptime:,48 secs
H2O_cluster_timezone:,Asia/Seoul
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,10 days
H2O_cluster_name:,H2O_from_python_tj_w4i9hp
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,12 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"


None


In [37]:
analyzer.train_automl(
    max_models=40,
    metric="RMSE",
    seed=23,
    nfolds=5
)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
17:02:42.471: AutoML: XGBoost is not available; skipping it.

██████████████████████████████████████████████████████████████ (cancelled) 100%


H2OJobCancelled: Job<$03017f00000132d4ffffffff$_8270cbf6d44efc8dd6e844bd8d1e4e62> was cancelled by the user.

In [ ]:
analyzer.use_stacked_ensemble_as_leader()

In [ ]:
analyzer.train_lgbm_optuna(n_trials=20)

In [ ]:
analyzer.analyze_shap(sample=3000)

In [ ]:
analyzer.predict_test("submission_h2o.csv")